# B-mode maps and angular power spectra (Figures 3 and 4)

Reproduces **Figure 3** (B-mode maps at 150 GHz before and after
deprojection) and **Figure 4** (TT, EE, TE and BB power spectra) of the
companion data paper, for one realization.

All FITS maps live on the data server; set `DEPROJ_DATA_ROOT` to the directory
that contains `maps/`, `params/`, `mask/` and `FG/` (see the README).  Figures
are written to `output/`; no data files are produced.


In [ ]:
import os
import pathlib
import sys

import numpy as np
import healpy as hp
import pymaster as nmt
import matplotlib.pyplot as plt

PKG = pathlib.Path('..').resolve()                 # repository root
if not (PKG / 'deproj_utils.py').exists():
    PKG = pathlib.Path('.').resolve()
sys.path.insert(0, str(PKG))
from deproj_utils import clean_bmap, iqu2teb, plot_maps

DATA = pathlib.Path(os.environ.get('DEPROJ_DATA_ROOT', '/sharefs/alicpt/users/doujiazheng/data/deproj'))
OUT = pathlib.Path('output')
OUT.mkdir(exist_ok=True)

FREQ, SIM = 150, 0
NSIDE = 1024
LMAX = 700               # bandpower lmax, also the lmax of the NMT fields
LMAX_SHT = 2 * NSIDE     # lmax of the spin-2 transforms
BIN_SIZE = 30
FWHM = {95: 19., 150: 11.}[FREQ]         # beam FWHM in arcmin

## Mask

The analysis mask is apodized with a 6 degree cosine (C2) profile for the NMT spectra; 
the B maps plotted use the binary mask.


In [ ]:
msk = hp.read_map(DATA / 'mask' / 'AliCPT_20uKcut150_C_1024.fits',
                   field=None, dtype=np.float64)
msk_apo = nmt.mask_apodization(msk, 6.0, apotype='C2')

## The three map versions

For realization `SIM` the dataset provides (i) the unconvolved CMB map
`LensInfo/CMB_LEN.fits`, which is smoothed to the band beam and added to the
foreground map `FG/group1_map_<FREQ>GHz.fits`, (ii) the map with beam
systematics `Ali_<FREQ>.fits`, and (iii) the deprojected map
`DEPROJ_Ali_<FREQ>.fits`.


In [ ]:
root = DATA / 'maps' / 'LensWithBeamSys' / str(SIM).zfill(3)
cmb = hp.read_map(root / 'LensInfo' / 'CMB_LEN.fits', field=None, dtype=np.float64)
fg = hp.read_map(DATA / 'FG' / ('group1_map_%dGHz.fits' % FREQ),
                 field=None, dtype=np.float64)

maps = {
    'CMB+FG, no beam systematics':
        hp.smoothing(cmb, fwhm=np.deg2rad(FWHM / 60.)) + fg,
    'CMB+FG, with beam systematics':
        hp.read_map(root / ('Ali_%d.fits' % FREQ), field=None, dtype=np.float64),
    'CMB+FG, deprojected':
        hp.read_map(root / ('DEPROJ_Ali_%d.fits' % FREQ), field=None, dtype=np.float64),
}

## Figure 3: B-mode maps

The B map is obtained with E-mode recycling to suppress the T-to-P leakage due to the partial-sky effect: 
the B map of the T+E part of the input is used as a template for the E-to-B leakage and is fitted 
and subtracted from the total B map inside the mask (`deproj_utils.clean_bmap`).


In [ ]:
bmap = {k: clean_bmap(v, msk, lmax=LMAX_SHT) for k, v in maps.items()}

plot_maps([bmap[k] * msk_apo for k in maps],
          titles=['no beam systematics', 'with beam systematics', 'deprojected'],
          vmin=-1., vmax=1., unit=r'$\mu$K',
          outfile=OUT / ('Bmaps_%dGHz.png' % FREQ));

## Figure 4: power spectra

Binned pseudo-Cl spectra are computed with NaMaster: a coupling matrix is
built for the apodized mask and inverted on the mode-coupling matrix, and the
result is divided by the Gaussian beam.  The bandpowers use a constant bin
size of 30 up to lmax = 700.


In [ ]:
binning = nmt.NmtBin.from_lmax_linear(LMAX, nlb=BIN_SIZE, is_Dell=True)
leff = binning.get_effective_ells()


def decoupled_dl(map1, map2):
    """Decoupled D_l of two maps with the band beam deconvolved."""
    f1 = nmt.NmtField(msk_apo, [map1], lmax=LMAX, masked_on_input=False)
    f2 = nmt.NmtField(msk_apo, [map2], lmax=LMAX, masked_on_input=False)
    wsp = nmt.NmtWorkspace()
    wsp.compute_coupling_matrix(f1, f2, binning)
    cl = nmt.compute_coupled_cell(f1, f2)
    beam = hp.gauss_beam(np.deg2rad(FWHM / 60.), lmax=LMAX)
    return wsp.decouple_cell(cl / beam ** 2)[0]


Dl = {}
for k, m in maps.items():
    t, e = iqu2teb(m, mask=msk, teb='te', lmax=LMAX_SHT)
    Dl[k] = {'TT': decoupled_dl(t, t), 'EE': decoupled_dl(e, e),
             'TE': decoupled_dl(t, e), 'BB': decoupled_dl(bmap[k], bmap[k])}

In [ ]:
keys = list(maps)
styles = ['k-', 'b-', 'ro-', 'c--']
labels = ['input CMB + FG', 'beam systematics', 'deprojected', 'deprojection residual']

fig, ax = plt.subplots(2, 2, figsize=(7.0, 4.32), dpi=300, sharex=True, tight_layout=True)
for a, pol in zip(ax.ravel(), ['TT', 'EE', 'TE', 'BB']):
    residual = Dl[keys[2]][pol] - Dl[keys[0]][pol]
    curves = [Dl[k][pol] for k in keys] + [residual if pol == 'TE' else abs(residual)]
    for curve, style, label in zip(curves, styles, labels):
        a.plot(leff[1:], curve[1:], style, lw=1., ms=1., alpha=0.7, label=label)
    a.text(0.05, 0.95, r'$%s$' % pol, ha='left', va='top', transform=a.transAxes)
    a.set_xlabel(r'$\ell$')
    a.set_ylabel(r'$\mathcal{D}^{%s}_\ell$ [$\mu$K$^2$]' % pol)
    a.set_xscale('log')
for a in [ax[0, 0], ax[0, 1], ax[1, 1]]:
    a.set_yscale('log')
ax[0, 0].legend(loc='best', frameon=False, fontsize=6)
fig.savefig(OUT / ('Dl_%dGHz.png' % FREQ), bbox_inches='tight', pad_inches=0.1)

The bandpowers and effective multipoles are kept in the dictionaries `Dl`
and the array `leff`.  The BB residual is two to three orders of magnitude
below the input BB spectrum, confirming that the deprojection suppresses the
T-to-B leakage to a negligible level.
